# Study report: scheduler

The study compares two run queues with the default `SCHED_SIMPLE` (a
sorted list): `scalable` (`SCHED_SCALABLE=y`, a red-black tree) and
`multiq` (`SCHED_MULTIQ=y`, one list for each priority). It also measures
the cost of 16 busy threads for the wake. The study imports the three
experiments of the parent study kconfig-factors.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-kconfig-factors@nucleo_g474re/st-scheduler"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Background

The IRQ entry path has no scheduler. The busy threads have priority 5,
lower than the woken thread. Thus the sorted insert of `SCHED_SIMPLE`
stops at once. With time slicing, the kernel slices the busy thread.
This adds one slice abort and one add for each wake (review of
2026-10-25, §5).

In [ ]:
LOAD_NAMES = {"none": "no load", "busy16": "16 busy threads", "t64": "64 load timers"}


def diffs(exp):
    """Each point minus the point of the variant baseline with the same load, in the same run (nbfig.diffs)."""
    return nbfig.diffs(inst, exp, "variant", "baseline", match=("load",), names=LOAD_NAMES)

## How to read the difference figures

Each bar is the value of a point minus the value of the point with the
variant `baseline` and the same load, in the same run. A bar to the left
of 0 means that the variant is faster. The three panels give the p50,
the p99 and the max. Each panel has its own x scale. The colour is the
load of the point. The menu above the figure selects the metric.

A grey band is the range that a check of this study expects, in the
same unit. If a band has one open end, it comes from an `order` check.
The manual checks are in the table of the checks only. The table below
each figure gives the same differences in cycles.

## H1: IRQ entry (irq-kconfig-1m)

H1 a and b expect no change of the `irq` p50 (+-2 cycles), because the
IRQ entry path has no scheduler.

In [ ]:
VARIANTS = ['scalable', 'multiq']
if nbfig.built(inst, "irq-kconfig-1m"):
    d = diffs("irq-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "irq-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "irq-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "irq-kconfig-1m") else None

## H1: ISR to thread (wake-kconfig-1m)

H1 c expects 50..200 cycles more for `scalable`. H1 e and f expect less
than 30 cycles of change for `multiq`, with no load and with 16 busy
threads. The points `*-busy16` have 16 busy threads.

In [ ]:
VARIANTS = ['scalable', 'multiq']
if nbfig.built(inst, "wake-kconfig-1m"):
    d = diffs("wake-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "wake-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "wake-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "wake-kconfig-1m") else None

## H1: thread to thread (thread-kconfig-1m)

H1 g expects 50..200 cycles more for `scalable`, H1 h less than 30
cycles of change for `multiq`.

In [ ]:
VARIANTS = ['scalable', 'multiq']
if nbfig.built(inst, "thread-kconfig-1m"):
    d = diffs("thread-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "thread-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "thread-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "thread-kconfig-1m") else None

## H1 d: the cost of the run queue with 16 busy threads

H1 d reads "more than with none" as a change of the cost of the run
queue from 0 to 16 busy threads. Each bar is (variant-busy16 -
baseline-busy16) - (variant - baseline), in the same run. The band is
the limit of H1 d (at most 40 cycles for `scalable`, p50).

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    d = diffs("wake-kconfig-1m")
    key = ["run", "tree", "rep", "variant", "metric"]
    none = d[d["axis.load"] == "none"].set_index(key)
    busy = d[d["axis.load"] == "busy16"].set_index(key)
    ch = busy[[]].copy()
    for s in nbfig.DIFF_STATS:
        ch["v_" + s] = busy["d_" + s]          # the difference with 16 busy threads
        ch["b_" + s] = none["d_" + s]          # the difference with no load
        ch["d_" + s] = ch["v_" + s] - ch["b_" + s]
    ch = ch.reset_index()
    ch = ch[ch.variant != "baseline"].assign(group="busy16 - no load")
    variant_of = d.drop_duplicates("point").set_index("point").variant
    lim = pd.DataFrame([{"check": f"{h['id']} {c['id']}", "variant": variant_of.get(c["select"]["point"]),
                         "metric": c["metric"], "stat": c["stat"], "expect": str(c["expect"]),
                         "lo": c["expect"]["diff_change"]["range"][0], "hi": c["expect"]["diff_change"]["range"][1]}
                        for h in inst.design_hypotheses.values() for c in h.get("checks", [])
                        if "diff_change" in c.get("expect", {})])
    lim["lo"] = lim.lo.fillna(-nbfig.INF)
    lim["hi"] = lim.hi.fillna(nbfig.INF)
    nbfig.diff_fig(ch, "wake-kconfig-1m: the change of the difference from no load to busy16", limits=lim, y="variant",
             color="group", what="change").show()

In [ ]:
nbfig.diff_table(ch, y="variant") if nbfig.built(inst, "wake-kconfig-1m") else None

## H2: the cost of 16 busy threads for the wake

Each bar is the value of a point with 16 busy threads minus the value of
the point of the same variant with no load, in the same run. H2 expects
about 400 cycles (`isr2thr`). H2 a is a manual check.

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    cost = busy[[]].copy()
    for s in nbfig.DIFF_STATS:
        cost["v_" + s] = busy["v_" + s]
        cost["b_" + s] = none["v_" + s]
        cost["d_" + s] = cost["v_" + s] - cost["b_" + s]
    cost = cost.reset_index().rename(columns={"variant": "run queue"})
    nbfig.diff_fig(cost, "wake-kconfig-1m: 16 busy threads minus no load", y="run queue", color="run queue",
             what="busy16 - no load").show()

In [ ]:
nbfig.diff_table(cost, y="run queue") if nbfig.built(inst, "wake-kconfig-1m") else None

### The cost over the number of busy threads (wake-threads-1m)

H2 says that the cost does not depend on the number of busy threads.
`wake-kconfig-1m` has 16 busy threads only. The experiment
`wake-threads-1m` of the study zephyr-versions has 2..64. Its busy
threads run a busy loop at the lowest priority. The figure gives the
`isr2thr` of each point `busy-n<N>` minus the point `none-n0` of the
same run. The diamond is the cost from `wake-kconfig-1m` (baseline). The
p50 has the larger marker.

In [ ]:
import plotly.graph_objects as go

if nbfig.built(inst, "wake-threads-1m"):
    wt = inst.rows("wake-threads-1m")
    wt = wt[(wt.role == "evidence") & (wt.metric == "isr2thr")]
    ref = wt[wt.point == "none-n0"].set_index(["run", "tree"])[list(nbfig.DIFF_STATS)]
    bt = wt[wt["axis.kind"] == "busy"].join(ref.add_prefix("ref_"), on=["run", "tree"]).sort_values("value.threads")
    fig = go.Figure()
    for k, s in enumerate(["p50", "p99"]):
        fig.add_trace(go.Scatter(x=bt["value.threads"], y=bt[s] - bt["ref_" + s], mode="lines+markers",
                                 name=f"wake-threads-1m {s}", line={"color": nbfig.color(k), "width": 2,
                                                                   "dash": [None, "dash"][k]},
                                 marker={"size": [11, 7][k]}, customdata=bt.point,
                                 hovertemplate="%{customdata}: %{y:+.0f} cycles<extra>" + s + "</extra>"))
        if nbfig.built(inst, "wake-kconfig-1m"):
            c = cost[(cost["run queue"] == "baseline") & (cost.metric == "isr2thr")]
            fig.add_trace(go.Scatter(x=[16] * len(c), y=c["d_" + s], mode="markers", name=f"wake-kconfig-1m {s}",
                                     marker={"symbol": "diamond", "size": [15, 9][k], "color": nbfig.color(k),
                                             "line": {"color": "#fcfcfb", "width": 1}},
                                     hovertemplate="baseline-busy16: %{y:+.0f} cycles<extra>" + s + "</extra>"))
    nbfig.style(fig, "isr2thr: busy threads minus no load threads", "busy threads", "cost (cycles)",
                xaxis_type="log", xaxis_tickvals=sorted(bt["value.threads"].unique())).show()
    out = bt.set_index("point")[["value.threads"]].assign(**{f"cost {s}": (bt[s] - bt["ref_" + s]).values
                                                             for s in nbfig.DIFF_STATS})
else:
    out = None
out